#  0. 라이브러리 불러오기

In [ ]:
import pandas as pd
import os
import numpy as np
from matplotlib import pyplot as plt
from datetime import datetime
from sklearn.preprocessing import MinMaxScaler, StandardScaler
import tensorflow as tf
import torch
import pickle
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Bidirectional, Dense, Attention, Concatenate, TimeDistributed, Dropout
from tensorflow.keras.losses import mean_squared_error
from tensorflow.keras.utils import plot_model
from tensorflow.keras.models import load_model
from sklearn.metrics import r2_score
import joblib

# 1-1. IQR scaling 함수

In [718]:
def IQR_scale(df, columns):
    Q1 = df[columns].quantile(q=0.25)
    Q3 = df[columns].quantile(q=0.75)
    IQR = Q3 - Q1
    filters = [((df[column] <= Q3[column] + 2.8 * IQR[column]) & (df[column] >= Q1[column] - 2.8 * IQR[column])) for column in columns]
    combined_filter = np.logical_and.reduce(filters)
    df_IQR = df[combined_filter]
    print(f"No. of {columns}: {len(df)}, IQR: {len(df_IQR)}")

    return df_IQR

# 1-2. Sequence 처리 함수

In [719]:
def to_sequences_x(x, window_size=1, sliding_step=1, step_topredict_model=1):
    x_values = []

    for i in range((len(x) - window_size - step_topredict_model) // sliding_step):
        x_values.append(x.iloc[(i*sliding_step) : (i*sliding_step + window_size)].values)
    return np.array(x_values)

def to_sequences_y(x, window_size=1, sliding_step=1, step_topredict_model=1):
    x_values = []
    y_values = []

    for i in range((len(x) - window_size - step_topredict_model) // sliding_step):
        x_values.append(x.iloc[(i*sliding_step) : (i*sliding_step + window_size)].values)
        y_values.append(x.iloc[(i*sliding_step + window_size) : (i*sliding_step + window_size + step_topredict_model)].values)

    # 만약 window_size와 step_topredict가 다르면, y_values에 0으로 채워진 열을 추가 (Padding)
    if window_size != step_topredict_model:
        n = x.shape[-1]
        padding_array = np.full([len(y_values), window_size - step_topredict_model, n], 10)
        return np.concatenate((padding_array, np.array(y_values)), axis=1)
    else:
        return np.array(y_values)
    # return np.array(y_values)

def from_sequences_x(x_values, sliding_step=1):
    original_data = []
    window_size = x_values.shape[1]

    for i in range(len(x_values)):
        original_data.extend(x_values[i])
        if i < len(x_values) - 1:
            original_data.extend([np.nan] * (sliding_step * window_size))

    return np.array(original_data)

# 1-3. Custom loss 정의

In [720]:
def custom_loss(y_true, y_pred):
    mask = tf.reduce_any(tf.not_equal(y_true, 10), axis=-1, keepdims=True)
    mask = tf.cast(mask, tf.float32)

    y_true_masked = tf.multiply(y_true, mask)
    y_pred_masked = tf.multiply(y_pred, mask)

    loss = mean_squared_error(y_true_masked, y_pred_masked)

    return tf.reduce_sum(loss) / tf.reduce_sum(mask)

# 1-4. 리샘플링 함수

In [721]:
def resampling(df, resam_term):
    df = df.reset_index(drop=False)
    df['Datetime'] = pd.to_datetime(df['Datetime'])
    df = df.set_index('Datetime')
    df = df.resample(rule=resam_term).mean()
    df = df.dropna()
    print(f"No. of resampled: {len(df)}")
    return df

# 2. 데이터 불러오기

In [722]:
def import_data(sheetname= 'Gosan', mode = 'FLUX'):
    # sheetname = 'Gosan' # life / industry

    worksheet = pd.read_excel(taglist_name, sheet_name = sheetname)
    worksheet = worksheet.dropna(axis=1).reset_index(drop=True)
    if sheetname == 'Gosan_mini':
        dir_base = f'./Rawdata_0815/KSSCADA.' #데이터가 있는 디렉토리 및 중복되는 파일명 처리
    else:
        dir_base = f'./Rawdata/KSSCADA.'

    if mode == 'FLUX':
        df = worksheet.loc[
            (worksheet['비고'] != 'NU') &
            (worksheet['사용'] == 'Y') &
            # (worksheet['데이터>200만'] == 'Y') &
            (worksheet['변수명'].str[0] != 'P')
            ].reset_index(drop=True)
    elif mode == 'PRES':
        df = worksheet.loc[
            (worksheet['비고'] != 'NU') &
            (worksheet['사용'] == 'Y') &
            # (worksheet['데이터>200만'] == 'Y') &
            (worksheet['변수명'].str[0] != 'Q')
            ].reset_index(drop=True)
    elif mode == 'BOTH':
        df = worksheet.loc[
            (worksheet['비고'] != 'NU') &
            (worksheet['사용'] == 'Y')
            # (worksheet['데이터>200만'] == 'Y')
            ].reset_index(drop=True)
    df_length = len(df)

    data_df = pd.DataFrame()
    print("Loading dataset. . .")

    try:
        data_df = joblib.load(f"{dir_base}{sheetname}_{mode}.pkl")
        print("Dataset loaded from pickle file.")
    except:
        print("Dataset pickle file not found. Loading from CSV files.")

        for i in range(df_length):
            try:
                print(f"{i+1}/{df_length}")
                tag = df.태그명[i]
                var = df.변수명[i]
                file_path = f"{dir_base}{tag}.F_CV.csv"

                # 우선 조건 필터링된 데이터 시도
                try:
                    temp = pd.read_csv(file_path, names=['Datetime', var, 'value2'])
                    temp = temp.query('value2 == 100').drop(columns='value2')
                    if len(temp) == 0:
                        raise Exception("No data after filtering by value2 == 100")
                except Exception:
                    # fallback: value2 컬럼이 없거나 실패 시
                    temp = pd.read_csv(file_path, names=['Datetime', var])

                temp = temp.set_index('Datetime').dropna()

                # IQR 스케일 적용
                scaled_df = IQR_scale(temp, [var])  # 반환값이 DataFrame이라 가정
                data_df[var] = scaled_df[var]

            except FileNotFoundError as e:
                print(f"[{tag}] File not found: {e}")
            except Exception as e:
                print(f"[{tag}] Error: {e}")

        # 저장
        with open(f"{dir_base}{sheetname}_{mode}.pkl", 'wb') as f:
            joblib.dump(data_df, f, compress=3)

    # try:
    #     data_df = joblib.load(f"{dir_base}{sheetname}_{mode}.pkl")
    #     print("Dataset loaded from pickle file.")
    # except:
    #     print("Dataset pickle file not found. Loading from CSV files.")
    #     try:
    #         for i in range(df_length):
    #             print(f"{i+1}/{df_length}")
    #             series = pd.Series(IQR_scale(pd.read_csv(f"{dir_base}{df.태그명[i]}.F_CV.csv", names=['Datetime', f"{df.변수명[i]}", 'value2']).query('value2 == 100').drop(columns='value2').set_index('Datetime').dropna(), [f"{df.변수명[i]}"]))
    #             if len(series) == 0:
    #                 series = pd.Series(IQR_scale(pd.read_csv(f"{dir_base}{df.태그명[i]}.F_CV.csv", names=['Datetime', f"{df.변수명[i]}"]).set_index('Datetime').dropna(), [f"{df.변수명[i]}"]))
    #         data_df[f"{df.변수명[i]}"] = series
    #     except FileNotFoundError as e:
    #         print(f"File not found: {e}. Please check the file paths and names.")
    #         raise Exception("Data loading failed. Please ensure the CSV files are available.")
        
    #     with open(f"{dir_base}{sheetname}_{mode}.pkl", 'wb') as f:
    #         joblib.dump(data_df, f, compress=3)

    # for i in range(df_length):
    #     print(f"{i+1}/{df_length}")
    #     # data_df[f"{df.변수명[i]}"] = pd.read_csv(f"{dir_base}{df.태그명[i]}.F_CV.csv", names = ['Datetime',f"{df.변수명[i]}",'value2']).drop(columns = 'value2').set_index('Datetime').dropna()
    #     data_df[f"{df.변수명[i]}"] = pd.read_csv(f"{dir_base}{df.태그명[i]}.F_CV.csv", names=['Datetime', f"{df.변수명[i]}", 'value2']).query('value2 == 100').drop(columns='value2').set_index('Datetime')

    # data_df = data_df.fillna(data_df.mean())
    data_df = data_df.dropna()

    # print(df)
    # print(data_df)
    print(data_df.shape)
    return df, data_df


# 3. 전처리

In [723]:
def preprocessing(df, data_df, del_var = False, show_map = True, scalers = {}):
    try:
        data_df = data_df.loc[data_df['Q_GS_NEW'] < 6000]
        data_df = data_df.loc[data_df['Q_GS_NEW'] > 0]
        data_df = data_df.loc[data_df['Q_GS_OLD'] > 14000]
        data_df = data_df.loc[data_df['H55_4'] > 2.5]
        data_df = data_df.loc[data_df['H56_4'] > 2.5]
        data_df = data_df.loc[data_df['H59_2'] > 1.9]
        data_df = data_df.loc[data_df['Q_GS_OLD'] > data_df['Q_GS_NEW']]
    except :
        pass
    data_df = data_df.interpolate(method='linear')

    data_df_resam = data_df.copy()
    data_df_resam = data_df_resam.reset_index(drop=False)
    data_df_resam['Datetime'] = pd.to_datetime(data_df_resam['Datetime'])
    # data_df_resam = data_df_resam.loc[(data_df_resam['Datetime'] >= '2023-01-01') & (data_df_resam['Datetime'] < '2024-10-15')].reset_index(drop=True)
    data_df_resam = data_df_resam.set_index('Datetime')
    data_df_resam = data_df_resam.resample(rule='1min').mean(numeric_only=True)
    # data_df_resam = data_df_resam.resample(rule='1min').first()
    # data_df_resam = data_df_resam.rolling(window=10).mean()
    data_df_resam = data_df_resam.dropna()
    print(data_df_resam)

    # df_to_sequence = data_df_resam.copy().reset_index(drop=True).fillna(data_df_resam.mean())
    df_to_sequence = data_df_resam.copy().reset_index(drop=True).dropna()

    if del_var:
        for col in df_to_sequence.columns:
            if df_to_sequence[col].nunique() < 0.5*len(df_to_sequence):
                # df_to_sequence.drop([col], axis=1, inplace=True)
                print(f"Warning: Variable {col} have some problem")
                print(f"length of data : {len(df_to_sequence)}")
                print(f"length of unique : {df_to_sequence[col].nunique()}")
                continue
        cal_corr(df, df_to_sequence, show_map = show_map)

    if scalers == {}:
        for feature in df.변수명.values:
            scaler = MinMaxScaler()
            scaler.fit(df_to_sequence[[feature]])
            df_to_sequence[feature] = scaler.transform(df_to_sequence[[feature]])
            scalers[feature] = scaler
    else:
        # print(scalers)
        for feature in df.변수명.values:
            df_to_sequence[feature] = scalers[feature].transform(df_to_sequence[[feature]])

    len_train = int(len(df_to_sequence)*0.8)

    trainset = df_to_sequence[:len_train]
    testset = df_to_sequence[len_train+1:]

    d_trainX = to_sequences_x(trainset[list(df.loc[df['비고'] != 'NU'].변수명)], window_size=window_size,sliding_step=sliding_step, step_topredict_model=step_topredict_model)
    d_trainY = to_sequences_y(trainset[list(df.loc[df['비고'] == 'target'].변수명)], window_size=window_size,sliding_step=sliding_step, step_topredict_model=step_topredict_model)

    print(f"Shape of trainX: {d_trainX.shape}")
    print(f"Shape of trainY: {d_trainY.shape}")

    d_testX = to_sequences_x(testset[list(df.loc[df['비고'] != 'NU'].변수명)], window_size=window_size, sliding_step=sliding_step, step_topredict_model=step_topredict_model)
    d_testY = to_sequences_y(testset[list(df.loc[df['비고'] == 'target'].변수명)], window_size=window_size,sliding_step=sliding_step, step_topredict_model=step_topredict_model)
    print(f"Shape of testX: {d_testX.shape}")
    print(f"Shape of testY: {d_testY.shape}")

    return d_trainX, d_trainY, d_testX, d_testY, scalers

# 5. Pearson correlation 계산

In [724]:
def cal_corr(df, df_to_sequence, show_map = False):
    corr_matrix = df_to_sequence.corr()
    if show_map:
        import seaborn as sns
        sns.clustermap(corr_matrix,
                       annot = True,      # 실제 값 화면에 나타내기
                       cmap = 'RdYlBu_r',  # Red, Yellow, Blue 색상으로 표시
                       vmin = -1, vmax = 1, #컬러차트 -1 ~ 1 범위로 표시
                       )
        plt.show()

    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), 1).astype(bool))
    ind_a, ind_b = np.where(abs(upper > 0.95))
    inds = []

    for i in range(len(ind_a)):
        # if inds != []:
        #     print(f"Pearson correlation of {df.변수명[ind_a[i]]} and {df.변수명[ind_b[i]]} is higher than 0.95")
        # else:
        #     break
        # df_to_sequence = df_to_sequence.drop([f"{df.변수명[ind_b[i]]}"], axis=1)
        # df = df.loc[df.변수명 != f"{df.변수명[ind_b[i]]}"]
        print(f"Pearson correlation of {df['변수명'][ind_a[i]]} and {df['변수명'][ind_b[i]]} is higher than 0.95")

    return

# 6. Model building 함수

In [ ]:
def building_model(d_trainX, d_trainY):
    # Input layer
    inputs = Input(shape=(None, d_trainX.shape[2]))
    # LSTM layer
    lstm_out = LSTM(64, activation='tanh', return_sequences=True)(inputs)
    lstm_out = Dropout(0.2)(lstm_out)
    # Self-Attention layer
    attention = Attention()([lstm_out, lstm_out, lstm_out])
    # Concatenate attention output with LSTM output
    context = Concatenate()([lstm_out, attention])
    # Bidirectional LSTM layer
    # lstm_out_bi = Bidirectional(LSTM(32, activation='tanh', return_sequences=True))(context)
    # lstm_out_bi = Dropout(0.2)(lstm_out_bi)
    lstm_out2 = LSTM(32, activation='tanh', return_sequences=True)(context)
    lstm_out_2 = Dropout(0.2)(lstm_out2)
    # TimeDistributed Dense layer
    # outputs = TimeDistributed(Dense(d_trainY.shape[2]))(lstm_out_bi)
    outputs = TimeDistributed(Dense(d_trainY.shape[2]))(lstm_out_2)
    # Define model
    model = Model(inputs=inputs, outputs = outputs)
    # model.compile(optimizer='adam', loss = custom_loss)
    model.compile(optimizer='adam', loss=custom_loss)
    # Print model summary
    model.summary()
    # Plot model
    plt.figure(figsize=(10, 5))
    plot_model(model)
    return model

# def building_model(d_trainX, d_trainY):
#     from tensorflow.keras.layers import Input, LSTM, Bidirectional, Dense, Dropout, BatchNormalization, TimeDistributed, Concatenate, RepeatVector, Attention
#     from tensorflow.keras.models import Model
#     from tensorflow.keras.optimizers import Adam

#     # Input layer
#     inputs = Input(shape=(window_size, d_trainX.shape[2]))

#     # LSTM layer with Dropout and BatchNormalization
#     lstm_out = LSTM(128, activation='tanh', return_sequences=True)(inputs)
#     lstm_out = Dropout(0.2)(lstm_out)
#     lstm_out = BatchNormalization()(lstm_out)

#     # Query vector
#     query = LSTM(128, activation='tanh', return_sequences=False)(inputs)
#     query = RepeatVector(window_size)(query)

#     # Attention layer
#     attention_out = Attention()([query, lstm_out])

#     # Concatenate attention output with LSTM output
#     context = Concatenate()([lstm_out, attention_out])

#     # Bidirectional LSTM layer with Dropout and BatchNormalization
#     lstm_out_bi = Bidirectional(LSTM(64, activation='tanh', return_sequences=True))(context)
#     lstm_out_bi = Dropout(0.2)(lstm_out_bi)
#     lstm_out_bi = BatchNormalization()(lstm_out_bi)

#     # TimeDistributed Dense layer with linear activation
#     outputs = TimeDistributed(Dense(d_trainY.shape[2], activation='linear'))(lstm_out_bi)

#     # Define model
#     model = Model(inputs=inputs, outputs=outputs)

#     # Compile model with adjusted learning rate and loss function
#     optimizer = Adam(learning_rate=0.001)
#     model.compile(optimizer=optimizer, loss=custom_loss)

#     # Print model summary
#     model.summary()

#     # Plot model (optional)
#     plot_model(model, show_shapes=True)

#     return model

In [726]:
# from tensorflow.keras.layers import Dropout, MultiHeadAttention
# def building_model(d_trainX, d_trainY):
#     # Input layer
#     inputs = Input(shape=(None, d_trainX.shape[2]))

#     # LSTM layer
#     lstm_out = LSTM(64, activation='tanh', return_sequences=True)(inputs)
#     lstm_out = Dropout(0.4)(lstm_out)

#     # Multihead Attention layer
#     attention = MultiHeadAttention(num_heads=4, key_dim=64)(lstm_out, lstm_out)

#     # Concatenate attention output with LSTM output
#     context = Concatenate()([lstm_out, attention])

#     # Bidirectional LSTM layer
#     lstm_out_bi = LSTM(64, activation='tanh', return_sequences=True)(context)
#     lstm_out_bi = Dropout(0.4)(lstm_out_bi)

#     # TimeDistributed Dense layer
#     outputs = TimeDistributed(Dense(d_trainY.shape[2]))(lstm_out_bi)

#     # Define model
#     model = Model(inputs=inputs, outputs=outputs)

#     # Model compile
#     model.compile(optimizer='adam', loss=custom_loss)

#     # Print model summary
#     model.summary()

#     # Plot model
#     plt.figure(figsize=(10, 5))
#     plot_model(model)

#     return model


In [727]:
# from keras import backend as K
# # Dual-Stage Attention Implementation
# # Dual-Stage Attention Implementation
# def dual_stage_attention(input_tensor):
#     # Extract input tensor shape
#     time_steps = tf.shape(input_tensor)[1]  # Get time_steps dynamically from input_tensor
#     features = input_tensor.shape[-1] if input_tensor.shape[-1] is not None else 64  # Default to 64 if None

#     # Step 1: Temporal Attention
#     # Apply attention across the time dimension
#     temporal_attention_scores = Dense(time_steps, activation='softmax')(input_tensor)
#     temporal_attention_scores = Lambda(lambda x: tf.expand_dims(x, axis=-1))(temporal_attention_scores)  # Expand dims to match input
#     temporal_context = Lambda(lambda x: tf.matmul(x[1], x[0], transpose_a=True))([temporal_attention_scores, input_tensor])

#     # Step 2: Variable Attention
#     # Apply attention across the variable dimension
#     variable_attention_scores = Dense(features, activation='softmax')(temporal_context)
#     variable_context = Lambda(lambda x: tf.reduce_sum(x[0] * x[1], axis=2))([variable_attention_scores, temporal_context])

#     # Add time dimension back to match lstm_out shape using Lambda layer
#     variable_context = Lambda(lambda x: tf.tile(tf.expand_dims(x, axis=1), [1, tf.shape(input_tensor)[1], 1]))(variable_context)

#     return variable_context

# def building_model(d_trainX, d_trainY):
#     # Input layer
#     inputs = Input(shape=(None, d_trainX.shape[2]))

#     # LSTM layer with Dropout
#     lstm_out = LSTM(64, activation='tanh', return_sequences=True)(inputs)
#     lstm_out = Dropout(0.3)(lstm_out)

#     # Dual-Stage Attention Mechanism
#     attention_context = dual_stage_attention(lstm_out)

#     # Concatenate attention output with LSTM output
#     context = Concatenate()([lstm_out, attention_context])

#     # Bidirectional LSTM layer with Dropout
#     lstm_out_bi = Bidirectional(LSTM(64, activation='tanh', return_sequences=True))(context)
#     lstm_out_bi = Dropout(0.3)(lstm_out_bi)

#     # TimeDistributed Dense layer
#     outputs = TimeDistributed(Dense(d_trainY.shape[2]))(lstm_out_bi)

#     # Define model
#     model = Model(inputs=inputs, outputs=outputs)

#     # Compile model
#     model.compile(optimizer='adam', loss=custom_loss)

#     # Print model summary
#     model.summary()

#     return model



# 7. Model training 함수

In [728]:
from keras.callbacks import Callback

class CustomEarlyStopping(Callback):
    def __init__(self, monitor='val_loss', min_epochs=10, patience=5, restore_best_weights=False):
        super(CustomEarlyStopping, self).__init__()
        self.monitor = monitor
        self.min_epochs = min_epochs
        self.patience = patience
        self.restore_best_weights = restore_best_weights
        self.best_weights = None
        self.best = np.Inf
        self.wait = 0
        self.stopped_epoch = 0

    def on_epoch_end(self, epoch, logs=None):
        current = logs.get(self.monitor)
        if current is None:
            return

        # 최소 에포크가 지나기 전에는 EarlyStopping을 무시
        if epoch < self.min_epochs:
            return

        # 개선되지 않으면 patience 카운트
        if np.less(current, self.best):
            self.best = current
            self.wait = 0
            if self.restore_best_weights:
                self.best_weights = self.model.get_weights()
        else:
            self.wait += 1
            if self.wait >= self.patience:
                self.stopped_epoch = epoch
                self.model.stop_training = True
                if self.restore_best_weights:
                    self.model.set_weights(self.best_weights)

    def on_train_end(self, logs=None):
        if self.stopped_epoch > 0:
            print(f"Early stopping at epoch {self.stopped_epoch + 1}.")

def training_model(model, d_trainX, d_trainY, epoch = 1000):
    from keras.callbacks import EarlyStopping
    custom_early_stopping = CustomEarlyStopping(monitor='val_loss', min_epochs=int(epoch*0.4), patience=20, restore_best_weights=True)
    # early_stopping = EarlyStopping(monitor='val_loss', patience = 20)
    model.fit(d_trainX, d_trainY, batch_size=256, epochs=epoch, validation_split = 0.1, callbacks = [custom_early_stopping])
    return model


# 8. 학습된 모델 저장

In [729]:
def save_trained_model(model, scalers, model_name='gs_v10',
                       base_path = "./"):
    # Define directory paths
    model_save_path = f"{base_path}saved_model/{model_name}.keras"
    model_save_path2 = f"{base_path}saved_model/{model_name}.h5"
    scaler_save_dir = f"{base_path}saved_scaler/{model_name}/"

    # Create the directory if it does not exist
    if not os.path.exists(scaler_save_dir):
        os.makedirs(scaler_save_dir)

    # Save model
    model.save(model_save_path)
    print(f"Model saved to {model_save_path}")

    model.save(model_save_path2)
    print(f"and saved to {model_save_path2}")

    # Save scalers
    for feature, scaler in scalers.items():
        save_path = os.path.join(scaler_save_dir, f"{feature}_scaler.pkl")  # 저장할 파일 경로
        with open(save_path, "wb") as f:
            pickle.dump(scaler, f)
    print(f"Model and scalers saved to {model_save_path}")

# 9. 학습된 모델 불러오기

In [730]:
def load_trained_model(df, model_name = 'gs_v10', base_path = "./"):
    try:
        print(f"Loading model from {base_path}saved_model/{model_name}.keras") # for debug
        model = load_model(f"{base_path}saved_model/{model_name}.keras", custom_objects={'custom_loss': custom_loss})
    except Exception as e:
        print(f"Error loading model: {e}")
        try:
            model = load_model(f"{base_path}saved_model/{model_name}.h5", custom_objects={'custom_loss': custom_loss})
        except Exception as e:
            print(f"Error loading model: {e}")

    scalers = {}
    for feature in df.loc[df['비고'] != 'NU'].변수명:
        scaler_path = f"{base_path}saved_scaler/{model_name}/{feature}_scaler.pkl"
        with open(scaler_path, "rb") as f:
            scalers[feature] = pickle.load(f)
    return model, scalers

# 10. 학습된 모델 평가(학습/테스트)

In [731]:
def evaluation_model_train(df, model, d_trainX, d_trainY):

    trainPredict = model.predict(d_trainX)

    trainAcc = []
    # beta = 0.5
    # betas = np.array([[(beta)**i] for i in range(step_topredict_user)])
    # betas = np.tile(betas, (d_trainY.shape[0], d_trainY.shape[2]))
    # recentY = d_trainY[:,-1,:]

    if step_topredict_user > 1:
        if step_topredict_model == step_topredict_user:
            trainPredict_result = trainPredict[:,-step_topredict_model:,:]
            # trainPredict_result = trainPredict_result*betas + recentY*(1-betas)
            trainY_result = d_trainY[:,-step_topredict_model:,:]

        else:
            trainPredict_result = trainPredict[:,-step_topredict_model:-step_topredict_model+step_topredict_user-1,:]
            # trainPredict_result = trainPredict_result*betas + recentY*(1-betas)
            trainY_result = d_trainY[:,-step_topredict_model:-step_topredict_model+step_topredict_user-1,:]

        for i in range(d_trainY.shape[2]):
            trainError = np.mean(abs(trainY_result[:,:,i] - trainPredict_result[:,:,i]), axis=1)
            trainMAE = np.mean(trainError)
            trainaccuracy = (1-trainMAE)*100
            r2 = r2_score(trainY_result[:,:,i], trainPredict_result[:,:,i])
            MSE = np.mean((trainY_result[:,:,i] - trainPredict_result[:,:,i])**2)
            RMSE = np.sqrt(MSE)
            trainAcc.append([trainMAE,trainaccuracy, MSE, r2])
            print(f'{df.변수명[i]} Train: MAE = {np.round(trainMAE,4)}, Accuracy = {np.round(trainaccuracy,1)}%')
    else:
        trainPredict_result = trainPredict[:,-step_topredict_model,:]
        # trainPredict_result = trainPredict_result*betas + recentY*(1-betas)
        trainY_result = d_trainY[:,-step_topredict_model,:]
        for i in range(d_trainY.shape[2]):
            trainMAE = np.mean(abs(trainY_result[:,i] - trainPredict_result[:,i]))
            trainaccuracy = (1-trainMAE)*100
            r2 = r2_score(trainY_result[:,i], trainPredict_result[:,i])
            MSE = np.mean((trainY_result[:,i] - trainPredict_result[:,i])**2)
            RMSE = np.sqrt(MSE)
            trainAcc.append([trainMAE,trainaccuracy, MSE, r2])
            print(f'{df.변수명[i]} Train: MAE = {np.round(trainMAE,4)}, Accuracy = {np.round(trainaccuracy,1)}%')
    print(trainAcc)
    return trainAcc, trainPredict_result, trainY_result
# print(f'Train MAE는 {np.round(trainMAE,4)}, 정확도는 {np.round(trainaccuracy,1)}% 입니다.')

def evaluation_model_test(df, model, d_testX, d_testY):
    testPredict = model.predict(d_testX)

    testAcc = []
    # beta = 0.5
    # betas = np.array([[(beta)**i] for i in range(step_topredict_user)])
    # betas = np.tile(betas, (d_testY.shape[0], d_testY.shape[2]))
    # recentY = d_testY[:,-1,:]

    if step_topredict_user > 1:
        if step_topredict_model == step_topredict_user:
            testPredict_result = testPredict[:,-step_topredict_model:,:]
            # testPredict_result = testPredict_result*betas + recentY*(1-betas)
            testY_result = d_testY[:,-step_topredict_model:,:]
        else:
            testPredict_result = testPredict[:,-step_topredict_model:-step_topredict_model+step_topredict_user-1,:]
            # testPredict_result = testPredict_result*betas + recentY*(1-betas)
            testY_result = d_testY[:,-step_topredict_model:-step_topredict_model+step_topredict_user-1,:]
        for i in range(d_testY.shape[2]):
            testError = np.mean(abs(testY_result[:,:,i] - testPredict_result[:,:,i]), axis=1)
            testMAE = np.mean(testError)
            testaccuracy = (1-testMAE)*100
            testAcc.append([testMAE,testaccuracy])
            print(f'{df.변수명[i]} Test: MAE = {np.round(testMAE,4)}, Accuracy = {np.round(testaccuracy,1)}%')
    else:
        testPredict_result = testPredict[:,-step_topredict_model,:]
        # testPredict_result = testPredict_result*betas + recentY*(1-betas)
        testY_result = d_testY[:,-step_topredict_model,:]
        for i in range(d_testY.shape[2]):
            testMAE = np.mean(abs(testY_result[:,i] - testPredict_result[:,i]))
            testaccuracy = (1-testMAE)*100
            testAcc.append([testMAE,testaccuracy])
            print(f'{df.변수명[i]} Test: MAE = {np.round(testMAE,4)}, Accuracy = {np.round(testaccuracy,1)}%')
    # print(testAcc)
    return testAcc, testPredict_result, testY_result
# print(f'Test MAE는 {np.round(testMAE,4)}, 정확도는 {np.round(testaccuracy,1)}% 입니다.')

# 11. 테스트 결과 plot

In [732]:
def plot_train_result(df, model, scalers, d_testX, d_testY, testY_result, testPredict_result, num_of_plot = 20, model_name = None):
    time_now = datetime.now().astimezone(KST).strftime('%Y-%m-%d %H:%M')

    n = num_of_plot  # Number of seeds
    num_targets = len(df.loc[df['비고'] == 'target'].변수명)  # Number of target variables
    fig_width = num_targets * 10
    fig_height = n * 5
    plt.figure(figsize=[fig_width, fig_height])

# Loop through each plot requirement
    subplot_index = 0
    for j in range(n):
        seed = np.random.randint(0, d_testX.shape[0])  # Random seed index
        variable_index = -1
        for i in range(num_targets):

            variable_index += 1
            subplot_index += 1  # Correct subplot index

            # Select the current target variable name
            target_var_name = list(df.loc[df['비고'] == 'target'].변수명)[variable_index]

            # Plot settings
            plt.subplot(n, num_targets, subplot_index)
            plt.plot(range(0, window_size), scalers[target_var_name].inverse_transform(d_testY[seed-window_size:seed, -step_topredict_model, variable_index].reshape(-1, 1)).flatten(), 'k-', label=target_var_name)

            # plt.plot(range(0, window_size), scalers[target_var_name].inverse_transform(d_testX[seed, :, variable_index].reshape(-1, 1)), 'k-', label=target_var_name)
            plt.plot(range(window_size, window_size + step_topredict_user), scalers[target_var_name].inverse_transform(testY_result[seed, :, variable_index].reshape(-1, 1)).flatten(), '-k^', label=f"{target_var_name}_GroundTruth")
            plt.plot(range(window_size, window_size + step_topredict_user), scalers[target_var_name].inverse_transform(testPredict_result[seed, :, variable_index].reshape(-1, 1)).flatten(), '-ro', label=f"{target_var_name}_Predict")

            plt.title(f"Seed: {seed}")
            # ymin, ymax = scalers[target_var_name].inverse_transform([[0], [1]]).flatten()
            # plt.ylim([ymin, ymax])
            plt.xlabel('Time (min)')
            plt.ylabel('Value')
            plt.legend()

    plt.tight_layout()  # Adjust layout to not overlap plots
    # plt.show()
    plt.savefig(f"./{model_name}_{time_now}_train.png", dpi = 100)
    plt.close()
    # plt.savefig('plot.png')
    return

def plot_test_result(df, model, scalers, d_testX, d_testY, testY_result, testPredict_result, num_of_plot = 20, model_name = None):
    time_now = datetime.now().astimezone(KST).strftime('%Y-%m-%d %H:%M')


    n = num_of_plot  # Number of seeds
    num_targets = len(df.loc[df['비고'] == 'target'].변수명)  # Number of target variables
    fig_width = num_targets * 10
    fig_height = n * 5

    plt.figure(figsize=[fig_width, fig_height])
    subplot_index = 0

# Loop through each plot requirement
    for j in range(n):
        seed = np.random.randint(0, d_testX.shape[0])  # Random seed index
        variable_index = -1
        for i in range(num_targets):
            variable_index += 1
            subplot_index += 1

            # Select the current target variable name
            target_var_name = list(df.loc[df['비고'] == 'target'].변수명)[variable_index]

            # Plot settings
            plt.subplot(n, num_targets, subplot_index)
            plt.plot(range(0, window_size), scalers[target_var_name].inverse_transform(d_testY[seed-window_size:seed, -step_topredict_model, variable_index].reshape(-1, 1)).flatten(), 'k-', label=target_var_name)

            # plt.plot(range(0, window_size), scalers[target_var_name].inverse_transform(d_testX[seed, :, variable_index].reshape(-1, 1)), 'k-', label=target_var_name)

            plt.plot(range(window_size, window_size + step_topredict_user), scalers[target_var_name].inverse_transform(testY_result[seed, :, variable_index].reshape(-1, 1)).flatten(), '-k^', label=f"{target_var_name}_GroundTruth")
            plt.plot(range(window_size, window_size + step_topredict_user), scalers[target_var_name].inverse_transform(testPredict_result[seed, :, variable_index].reshape(-1, 1)).flatten(), '-ro', label=f"{target_var_name}_Predict")


            plt.title(f"Seed: {seed}")
            # ymin, ymax = scalers[target_var_name].inverse_transform([[0], [1]]).flatten()
            # plt.ylim([ymin, ymax])
            plt.xlabel('Time (min)')
            plt.ylabel('Value')
            plt.legend()

    plt.tight_layout()  # Adjust layout to not overlap plots
    # plt.show()
    plt.savefig(f"./{model_name}_{time_now}_test.png", dpi = 100)
    plt.close()
    # plt.savefig('plot.png')
    return

# 복합함수

In [733]:
def import_and_preprocessing(sheetname = 'Gosan', mode = 'FLUX', del_var = False, show_map = True, scalers = {}):

    df, data_df = import_data(sheetname = sheetname, mode = mode)
    d_trainX, d_trainY, d_testX, d_testY, scalers = preprocessing(df, data_df, del_var = del_var, show_map = show_map, scalers = scalers)

    return df, d_trainX, d_trainY, d_testX, d_testY, scalers

def building_and_training_model(d_trainX, d_trainY, epoches = 1000):
    model = building_model(d_trainX, d_trainY)
    model = training_model(model, d_trainX, d_trainY, epoch = epoches)
    return model

def save_and_evaluate_model(df, model, scalers, d_trainX, d_trainY, d_testX, d_testY,
                            save_ = True, plot_ = True, train_ = True, test_ = True,
                            model_name = 'gs_v10'):
    df_result_summary = pd.DataFrame(columns=['모델명','태그명','태그 설명', '변수명','TrainMAE','TrainAccuracy', 'TestMAE', 'TestAccuracy'])

    df_result_summary['태그명'] = df.loc[df['비고'] == 'target'].태그명
    df_result_summary['태그 설명'] = df.loc[df['비고'] == 'target']['태그 설명'].values
    df_result_summary['변수명'] = df.loc[df['비고'] == 'target'].변수명
    df_result_summary['모델명'] = model_name

    if save_:
        save_trained_model(model, scalers, model_name = model_name, base_path = f"./")
    if train_:
        trainResult, trainPredict_result, trainY_result = evaluation_model_train(df, model, d_trainX, d_trainY)
        df_result_summary['TrainMAE'] = np.array(trainResult)[:,0]
        df_result_summary['TrainAccuracy'] = np.array(trainResult)[:,1]
    if test_:
        testResult, testPredict_result, testY_result = evaluation_model_test(df, model, d_testX, d_testY)
        df_result_summary['TestMAE'] = np.array(testResult)[:,0]
        df_result_summary['TestAccuracy'] = np.array(testResult)[:,1]
    if plot_:
        if step_topredict_user > 1:
            plot_train_result(df, model, scalers, d_trainX, d_trainY, trainY_result, trainPredict_result, num_of_plot = 20, model_name = model_name)
            plot_test_result(df, model, scalers, d_testX, d_testY, testY_result, testPredict_result, num_of_plot = 20, model_name = model_name)
        else:
            trainY_result = np.expand_dims(trainY_result, axis=1)
            trainPredict_result = np.expand_dims(trainPredict_result, axis=1)
            testY_result = np.expand_dims(testY_result, axis=1)
            testPredict_result = np.expand_dims(testPredict_result, axis=1)
            plot_train_result(df, model, scalers, d_trainX, d_trainY, trainY_result, trainPredict_result, num_of_plot = 20, model_name = model_name)
            plot_test_result(df, model, scalers, d_testX, d_testY, testY_result, testPredict_result, num_of_plot = 20, model_name = model_name)

    # if train_:
    #     if test_:
    #         return trainResult, trainPredict_result, trainY_result, testResult, testPredict_result, testY_result, df_result_summary
    #     else:
    #         return trainResult, trainPredict_result, trainY_result, df_result_summary
    # else:
    #     if test_:
    #         return testResult, testPredict_result, testY_result, df_result_summary
    #     else:
    return df_result_summary



In [734]:
def import_train_save_model(model_name = 'gs_v10', sheetname='Gosan',
                            mode = 'FLUX',
                            del_var = False,
                            show_map = True,
                            training = True,
                            save_ = True,
                            plot_ = False,
                            train_ = True,
                            test_ = True,
                            epoches = 1000
                            ):

    if training:
        df, d_trainX, d_trainY, d_testX, d_testY, scalers = import_and_preprocessing(sheetname = sheetname, mode = mode, del_var = del_var, show_map = show_map, scalers = {})
        model = building_and_training_model(d_trainX, d_trainY, epoches = epoches)

    else:
        df, data_df = import_data(sheetname = sheetname, mode = mode)
        model, scalers = load_trained_model(df, model_name = model_name, base_path = f"./")
        d_trainX, d_trainY, d_testX, d_testY, scalers = preprocessing(df, data_df, del_var = del_var, show_map = show_map, scalers = scalers)

    result_summary = save_and_evaluate_model(df, model, scalers, d_trainX, d_trainY, d_testX, d_testY,
                        save_ = save_, plot_ = plot_, train_ = train_, test_ = test_,
                        model_name = model_name)
    return result_summary

def RELU(x):
    return np.maximum(0, x)

# 디버그 실행

In [ ]:

import openpyxl
import pytz

def main_full():
    global window_size, sliding_step, step_topredict_model, step_topredict_user, KST, taglist_name
    window_size = 60
    sliding_step = 1
    step_topredict_model = 10
    step_topredict_user = 10
    KST = pytz.timezone('Asia/Seoul')

    # mode: pressure data를 쓸지 안쓸지 결정(FLUX: 유량만, PRES: 압력만, BOTH: 모두)
    # del_var: Pearson 계수 기반 중복데이터를 출력할지 결정
    # show_map: Pearson correlation map을 출력할지 결정
    # training: 모델을 새로 학습할지 기존 모델을 불러올지 결정 (False : 기존 모델 불러오기)
    # save_: 모델을 저장할 지 결정
    # plot_: 테스트결과를 출력할지 결정
    # train_: 학습데이터 정확도를 계산할지 결정
    # test_: 테스트데이터 정확도를 계산할지 결정

    taglist_name = "./GS_taglist_backup.xlsx"
    worksheet = openpyxl.load_workbook(taglist_name)

    result_summary = pd.DataFrame(columns=['모델명','태그명','태그 설명', '변수명','TrainMAE','TrainAccuracy', 'TestMAE', 'TestAccuracy'])

    for sheetname in worksheet.sheetnames[1:]:
        print(f"Training model {sheetname}")
        if sheetname == 'Gosan_mini':
            result_summary = pd.concat(
                [result_summary, import_train_save_model(model_name = f"gs_flux_test", sheetname = sheetname,
                            mode = 'FLUX',
                            del_var = False,
                            show_map = True,
                            training = True,
                            save_ = True,
                            train_ = True, test_ = True,
                            plot_ = True,
                            epoches = 1000)], axis = 0
            ).reset_index(drop=True)
            result_summary = pd.concat(
                [result_summary, import_train_save_model(model_name = f"gs_pres_test", sheetname = sheetname,
                            mode = 'PRES',
                            del_var = False,
                            show_map = True,
                            training = True,
                            save_ = True,
                            train_ = True, test_ = True,
                            plot_ = True,
                            epoches = 1000)], axis = 0
            ).reset_index(drop=True)
        else:
            result_summary = pd.concat(
                [result_summary, import_train_save_model(model_name = f"{sheetname}_test", sheetname = sheetname,
                                mode = 'BOTH',
                                del_var = False,
                                show_map = True,
                                training = True,
                                save_ = True,
                                train_ = True, test_ = True,
                                plot_ = True,
                                epoches = 1000)], axis = 0
            ).reset_index(drop=True)
    return result_summary

In [ ]:
if __name__ == "__main__":
    result = main_full()
    result.to_excel(f"./GS_model_result_{datetime.now().strftime('%Y%m%d_%H%M%S')}.xlsx", index = False)

Training model Gosan_mini
Training model gs
Loading dataset. . .
Dataset pickle file not found. Loading from CSV files.
1/4
No. of ['Q1']: 2624769, IQR: 2093838
2/4
No. of ['Q2']: 2624769, IQR: 2095999
3/4
No. of ['P1']: 53442, IQR: 53120
4/4
No. of ['H2_1']: 2624769, IQR: 2592332
(47676, 4)
                        Q1     Q2     P1      H2_1
Datetime                                          
2022-11-24 20:47:00  122.0  102.5  6.780  3.685387
2022-11-24 20:48:00  122.0  102.4  6.780  3.686156
2022-11-24 20:49:00  122.0  102.4  6.780  3.685387
2022-11-24 20:50:00  122.0  102.9  6.780  3.686156
2022-11-24 20:51:00  122.0  102.4  6.780  3.685387
...                    ...    ...    ...       ...
2022-12-31 23:55:00  133.0  112.3  7.110  3.348162
2022-12-31 23:56:00  136.0  112.9  7.115  3.347650
2022-12-31 23:57:00  134.0  112.8  7.115  3.347650
2022-12-31 23:58:00  133.0  113.1  7.120  3.348162
2022-12-31 23:59:00  135.0  112.6  7.115  3.347394

[47676 rows x 4 columns]
Shape of trainX: (

KeyboardInterrupt: 

<Figure size 1000x500 with 0 Axes>

<Figure size 1000x500 with 0 Axes>